# plumbline — quickstart

This notebook is for a *consumer* of the framework: you built a text-to-SQL
system and want to grade it. Every cell here runs offline and free — SQLite,
no cloud, no API keys.

Run it from the `notebook/` folder. Design reasons and module internals are in
[internals.ipynb](internals.ipynb); this notebook only shows what you do:

```
build a demo DB -> score predictions -> gate the answer key -> score routing
```

In [ ]:
import subprocess


def sh(command):
    """Run a shell command, print it like a terminal session, return output."""
    print(f"$ {command}")
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if output:
        print(output)
    return output


def show(table):
    """Display a pandas table in Jupyter; plain-print under exec()-based tests."""
    try:
        from IPython.display import display

        display(table)
    except ImportError:
        print(table)


# Import the package straight from src/, so the notebook runs from a bare
# checkout with no install step.
import sys

sys.path.insert(0, "../src")

## Step 1 — build the demo database

The framework grades SQL by **running** it, not by reading it, so a database
comes first. This builds a tiny employee/department SQLite file. Free.

In [ ]:
sh("cd .. && uv run --no-project python examples/build_db.py")

## Step 2 — score predictions

The input is one CSV. `gold_sql` is the reference answer (the answer key);
`pred_sql` is what your system produced for the same question. Look at the
real file first:

In [ ]:
sh("cd .. && cat examples/predictions.csv")

In [ ]:
sh("cd .. && uv run --no-project python -m plumbline score"
   " --db examples/demo.db --predictions examples/predictions.csv")

How to read this report:

- `execution accuracy` — the share of predictions whose result **rows** equal
  the gold's rows. Differently written SQL that returns the same rows counts
  as correct.
- `95% CI [low, high]` — a confidence interval: the range the true rate could
  be in, given only this many questions. A wide range means "too few questions
  to conclude".
- `judge-adjusted` — the same rate after an optional LLM judge re-checks
  near-misses. The judge is off here, so the two rates are equal.
- per-question verdicts: `MATCH` / `MISMATCH` are the comparator's calls;
  `PRED_ERROR` means the predicted SQL crashed (counted wrong);
  `EMPTY_BOTH` is **excluded** — both queries returned nothing, which says
  nothing about the prediction, so it leaves the denominator entirely.

## Step 3 — gate the answer key

Every score is computed *against* the gold SQL, so a broken gold silently
corrupts the number — in the worst case it makes a wrong prediction look
right. The gate audits the golds themselves, and exits non-zero so it can
block a CI pipeline. The example golds are deliberately broken to show each
defect class.

In [ ]:
sh("cd .. && uv run --no-project python -m plumbline gate"
   " --db examples/demo.db --golds examples/golds.csv; echo \"gate exit code: $?\"")

Look at `gate exit code: 1` — that non-zero exit is the point: score a
gold set only after the gate reports it clean. Each flagged line names the
defect (`NON_SELECT`, `EXEC_FAIL`, `ZERO_ROWS`, `DEGENERATE`, ...) and a
one-line reason.

## Step 4 — routing accuracy

In a warehouse with many tables, a pipeline usually **routes** first: pick the
one `dataset.table` that should answer the question, then write SQL against
it. `score-routing` grades that stage alone (no database needed — it compares
table names). The CSV columns are `id,question,gold_table,pred_table`.

Two runs below: plain, then with `--catalog` — a text file listing every real
table, which lets the scorer *snap* a near-miss name (like a dropped plural)
onto the closest real one.

In [ ]:
sh("cd .. && uv run --no-project python -m plumbline score-routing"
   " --predictions examples/routing_predictions.csv")
print()
sh("cd .. && uv run --no-project python -m plumbline score-routing"
   " --predictions examples/routing_predictions.csv"
   " --catalog examples/routing_catalog.txt")

Compare the two runs:

- `dataset accuracy` — the part before the dot was right. `table accuracy` —
  the exact table was right. Most routing misses land in the right dataset,
  which is why both tiers are shown.
- Only row `r5` changed: `inventory.stock_level -> inventory.stock_levels
  snapped, correct`, lifting table accuracy from 3/6 to 4/6.
- The line `snapped 1 of 6 predictions to the catalog` is deliberate: a
  snapped rate is a *corrected* rate, and the report always says so. No
  snap line at all means snapping was off.
- An empty prediction (`r6`) counts as wrong — it never leaves the
  denominator.

**Explore it yourself:** open `examples/routing_predictions.csv` and
`examples/routing_catalog.txt` side by side in the file browser. Find the r4
prediction in the catalog — it is a *real* table, just the wrong pick, which
is why it did not snap.

## Step 5 — the same calls as a library

For your own pipeline or CI, skip the CLI and call the functions. The report
objects carry the same numbers the CLI printed.

In [ ]:
from plumbline.executor import SQLiteExecutor
from plumbline.harness import evaluate

executor = SQLiteExecutor("../examples/demo.db")
predictions = [
    ("p1", "how many employees?",
     "SELECT COUNT(*) FROM employee",     # gold
     "SELECT COUNT(id) FROM employee"),   # different SQL, same rows -> correct
]
report = evaluate(executor, predictions)
print(report.execution)

That `Proportion(k=1, n=1, point=1.0, low=..., high=...)` is the rate with
its confidence interval — the same object behind the CLI's first line. Note
how wide the interval is at n=1: one question proves almost nothing.

In [ ]:
from plumbline.routing import evaluate_routing

routing_report = evaluate_routing(
    [("r1", "orders this month?", "sales.orders", "analytics-prod.sales.orders")]
)
print(routing_report.results[0].pred_table)   # 3-part name normalized to 2-part
print(routing_report.table)

## Where next

- [README](../README.md) — the short overview.
- [docs/METHODOLOGY.md](../docs/METHODOLOGY.md) — why each metric is built
  this way, and the known limits.
- [internals.ipynb](internals.ipynb) — module-by-module deep dive.

Reproduce everything (all free): `make test`, `make example`,
`make example-routing`. The only paid path in the framework is
`score --judge-gemini`, which makes one Gemini call per disagreement.